In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# HF 空间软分配：三条件开发入口

代码已实现，真实模型尚未运行。本入口默认 `plan`；`fit` / `validation` 会加载 SD3.5、DINO、VAE、SyncSeal 和 LPIPS，需要另行获得本轮外部执行授权。

先将随分支交付的 `survival_allocator_v1.zip` 上传到下述路径，或把已解包代码放到 REPO。分支尚未推送，不能从远端 main 获得本次方法代码。

fit8：56张最终图、168条评分路径、32条HF宏块续采样标签。validation24：96张最终图、288条评分路径。当前仅clean、RGB[0,1] AWGN σ.02后clip、JPEG50；旧264图/3528路径计划已撤回。每个marked变体独立执行20步，在step18嵌入并真实执行step19；不是64个decode/encode探针。所有分数失败保留；无正式FPR结论。


In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, datetime
MODE = 'plan'  # plan / fit / validation
REPO = Path('/content/survival-allocator-v1')
SOURCE_ARCHIVE = Path('/content/drive/MyDrive/CEG-WM/development/survival_allocator_v1.zip')
if not REPO.exists():
    shutil.unpack_archive(str(SOURCE_ARCHIVE), '/content')
STAGE = 'validation' if MODE == 'validation' else 'fit'
ROSTER = REPO / 'configs/parallel_method_dev' / (STAGE + '.json')
OUTPUT = Path('/content') / ('allocator-' + STAGE + '-' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S'))
RUNTIME = Path('/content/ceg-method-models')
ALLOCATOR = Path('/content/allocator-fit/allocator.json')  # validation: set to actual fit output
print({'mode': MODE, 'roster': str(ROSTER), 'output': str(OUTPUT)})


## 环境与凭据
环境版本只记录。默认plan不读取凭据、不加载模型；真实运行需要GPU内存满足SD3.5 medium，不限定A100。输出先写Colab本地。

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(REPO), 'scipy', 'accelerate', 'lpips', 'torchmetrics'], check=True)
if MODE != 'plan':
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['CEG_WM_ROOT_KEY'] = userdata.get('CEG_WM_ROOT_KEY')
import importlib.metadata as metadata
print({name: metadata.version(name) for name in ['torch', 'diffusers', 'transformers', 'scipy']})


In [ ]:
command = [sys.executable, '-m', 'experiments.run_survival_allocator_dev',
           '--mode', MODE, '--roster', str(ROSTER), '--output', str(OUTPUT),
           '--runtime-root', str(RUNTIME)]
if MODE == 'validation':
    command += ['--allocator', str(ALLOCATOR)]
subprocess.run(command, cwd=REPO, check=True)


## 读取结果

真实执行后查看 `report.json`、`rows.jsonl`、`labels.jsonl` 和保存图像。fit生成 `allocator.json`；把其路径填入ALLOCATOR再单独运行validation。效用标签为当前三条件平均内容证据增量减最终LPIPS增量成本；LF参数与该图原LF/HF份额固定，只改变HF空间权重。实际联合/LF/HF扰动、PSNR、SSIM、LPIPS及2×2局部MSE分别报告；同时查看正负分离和负样本上尾。质量不匹配时不得声称同质量获益，允许均匀分配胜出。第三输入特征是latent能量，未实现完整生成Jacobian或响应稳定性。

本notebook仅完成结构及Python语法验证，Colab挂载、依赖安装、模型加载和真实输出仍未验证。外部执行授权后按单元顺序运行即可核查这些缺口。